<a href="https://colab.research.google.com/github/Balaji19Bt/bcl11a-sgrna-designer/blob/main/sgRNA_BCL11A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.0 MB/s eta 0:00:00


In [ ]:
from Bio.Seq import Seq
from Bio import SeqIO, Entrez

Entrez.email = "balajisridharan2007@gmail.com"

handle = Entrez.efetch(db="nucleotide", id="NG_042065", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
sequence = str(record.seq)

print("Sequence length:", len(sequence))
print("First 100 bases:", sequence[:100])

Sequence length: 23855
First 100 bases: GCCCAGCGTATTTACCATTTTCAAATTGATTTAAATAGTAACTTCCATAAGAAGATTAGAATTGCCCTGCTGGCTACTTTCTCTGATAAAAAAAATTTTT


In [ ]:
def find_sgRNAs(seq, pam="GG"):
    candidates = []
    for i in range(len(seq) - 22):
        window = seq[i:i+23]
        if window[21:23] == pam:
            guide = window[0:20]
            candidates.append((i, guide, window[20:23]))
    return candidates

results = find_sgRNAs(sequence)
print(f"Found {len(results)} candidate sgRNAs")
for pos, guide, pam in results[:5]:
    print(f"Position {pos}: guide={guide}, PAM={pam}")

Found 1421 candidate sgRNAs
Position 50: guide=GAAGATTAGAATTGCCCTGC, PAM=TGG
Position 144: guide=TATTAAAAGCCATGTGAAAA, PAM=AGG
Position 237: guide=CTATGTCCCAGCATCATGCC, PAM=AGG
Position 240: guide=TGTCCCAGCATCATGCCAGG, PAM=TGG
Position 245: guide=CAGCATCATGCCAGGTGGTA, PAM=AGG


In [ ]:
def gc_content(seq):
    return (seq.count("G") + seq.count("C")) / len(seq) * 100

filtered = [(pos, guide, pam) for pos, guide, pam in results if 40 <= gc_content(guide) <= 60]
print(f"{len(filtered)} sgRNAs passed GC filter (out of {len(results)})")

for pos, guide, pam in filtered[:5]:
    print(f"Position {pos}: guide={guide}, GC%={gc_content(guide):.1f}")

741 sgRNAs passed GC filter (out of 1421)
Position 50: guide=GAAGATTAGAATTGCCCTGC, GC%=45.0
Position 237: guide=CTATGTCCCAGCATCATGCC, GC%=55.0
Position 240: guide=TGTCCCAGCATCATGCCAGG, GC%=60.0
Position 245: guide=CAGCATCATGCCAGGTGGTA, GC%=55.0
Position 246: guide=AGCATCATGCCAGGTGGTAA, GC%=50.0


In [ ]:
# Get top 10 candidates closest to ideal 50% GC
filtered_sorted = sorted(filtered, key=lambda x: abs(gc_content(x[1]) - 50))
top10 = filtered_sorted[:10]

# Check how many times each guide's core sequence appears in the full region (uniqueness proxy)
def count_occurrences(seq, guide):
    return seq.count(guide)

print("Top 10 candidate sgRNAs with uniqueness check:\n")
for pos, guide, pam in top10:
    occurrences = count_occurrences(sequence, guide)
    print(f"Position {pos}: guide={guide}, GC%={gc_content(guide):.1f}, occurrences_in_region={occurrences}")

Top 10 candidate sgRNAs with uniqueness check:

Position 246: guide=AGCATCATGCCAGGTGGTAA, GC%=50.0, occurrences_in_region=1
Position 290: guide=TCAAGGAGCTCAGCCTAGTT, GC%=50.0, occurrences_in_region=1
Position 422: guide=ATCACAGGGGATGTGATGCT, GC%=50.0, occurrences_in_region=1
Position 423: guide=TCACAGGGGATGTGATGCTT, GC%=50.0, occurrences_in_region=1
Position 461: guide=TGTTCAGCAGGCAGTGGAAA, GC%=50.0, occurrences_in_region=1
Position 509: guide=AGCAGCACACGCAAAGATGT, GC%=50.0, occurrences_in_region=1
Position 515: guide=ACACGCAAAGATGTGGGAGT, GC%=50.0, occurrences_in_region=1
Position 544: guide=CACAGCACATCCAGATACCT, GC%=50.0, occurrences_in_region=1
Position 565: guide=GGAGTGGTTCAGTGTAGCTT, GC%=50.0, occurrences_in_region=1
Position 578: guide=GTAGCTTTGGTGACAGAGTG, GC%=50.0, occurrences_in_region=1


In [ ]:
import pandas as pd

df = pd.DataFrame(top10, columns=["Position", "Guide_Sequence", "PAM"])
df["GC_content(%)"] = df["Guide_Sequence"].apply(gc_content)
df["Occurrences_in_region"] = df["Guide_Sequence"].apply(lambda g: count_occurrences(sequence, g))
df = df.sort_values("GC_content(%)", ascending=False).reset_index(drop=True)

df

,Position,Guide_Sequence,PAM,GC_content(%),Occurrences_in_region
0,246,AGCATCATGCCAGGTGGTAA,GGG,50.0,1
1,290,TCAAGGAGCTCAGCCTAGTT,GGG,50.0,1
2,422,ATCACAGGGGATGTGATGCT,TGG,50.0,1
3,423,TCACAGGGGATGTGATGCTT,GGG,50.0,1
4,461,TGTTCAGCAGGCAGTGGAAA,AGG,50.0,1
5,509,AGCAGCACACGCAAAGATGT,GGG,50.0,1
6,515,ACACGCAAAGATGTGGGAGT,GGG,50.0,1
7,544,CACAGCACATCCAGATACCT,GGG,50.0,1
8,565,GGAGTGGTTCAGTGTAGCTT,TGG,50.0,1
9,578,GTAGCTTTGGTGACAGAGTG,AGG,50.0,1
